In [1]:
"""
10_xgboost_behavioural.py

Same setup as 09_xgboost_financial.py, financial ratios plus
behavioural variables (laat_dummy, boete_dummy, dgrmovestreet).
Train/test split is by bookyear (2018 train, 2019 test) - see
utils/time_based_split.py. See 11_train_window_sensitivity.ipynb for
the check on whether adding 2017 as a second training year helps.
"""


'\n10_xgboost_behavioural.py\n\nSame setup as 09_xgboost_financial.py, financial ratios plus\nbehavioural variables (laat_dummy, boete_dummy, dgrmovestreet).\nTrain/test split is by bookyear (2018 train, 2019 test) - see\nutils/time_based_split.py. See 11_train_window_sensitivity.ipynb for\nthe check on whether adding 2017 as a second training year helps.\n'

In [2]:
from utils.load_data_features import load_data_features

df = load_data_features()


In [3]:
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from config import (
    FEATURES_BEHAVIORAL, FEATURES_FINANCIAL, RANDOM_STATE, TARGET,
    WINSOR_COLUMNS,
)
from utils.model_results import save_model_results
from utils.print_section import print_section
from utils.time_based_split import time_based_split
from utils.winsorizer import Winsorizer

MODEL_NAME = "XGBoost Financial + Behavioral"
FEATURES = FEATURES_FINANCIAL + FEATURES_BEHAVIORAL

X_train, X_test, y_train, y_test = time_based_split(df, FEATURES, TARGET)

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()
scale_pos_weight = negative_class / positive_class

print_section("Class imbalance")
print(f"Negative class: {negative_class:,}")
print(f"Positive class: {positive_class:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

pipeline = Pipeline([
    ("winsorizer", Winsorizer(columns=WINSOR_COLUMNS)),
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    )),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print_section("Performance")

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob),
}

for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

print_section("Confusion matrix")

cm = confusion_matrix(y_test, y_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print()
print(f"True Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

print_section("Classification report")
print(classification_report(y_test, y_pred, zero_division=0))

print_section("Feature importance")

importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": pipeline.named_steps["model"].feature_importances_,
}).sort_values(by="importance", ascending=False)

print(importances)

print_section("Model improvement over baseline")

baseline_rate = y_train.mean()
print(f"Baseline failure rate (train): {baseline_rate:.4%}")
print(f"Model precision: {metrics['precision']:.4%}")
print(f"Improvement factor: {metrics['precision'] / baseline_rate:.2f}x")

save_model_results(MODEL_NAME, metrics)



Class imbalance
Negative class: 335,278
Positive class: 1,079
scale_pos_weight: 310.73



Performance


accuracy  : 0.8476
precision : 0.0088
recall    : 0.7076
f1        : 0.0174
roc_auc   : 0.8586
pr_auc    : 0.0233

Confusion matrix
[[275130  49360]
 [   181    438]]

True Negatives : 275,130
False Positives: 49,360
False Negatives: 181
True Positives : 438

Classification report
              precision    recall  f1-score   support

           0       1.00      0.85      0.92    324490
           1       0.01      0.71      0.02       619

    accuracy                           0.85    325109
   macro avg       0.50      0.78      0.47    325109
weighted avg       1.00      0.85      0.92    325109


Feature importance
         feature  importance
8  dgrmovestreet    0.192748
2       solvency    0.184095
0  profitability    0.127523
3      structure    0.103574
7    boete_dummy    0.089459
1      liquidity    0.089058
5           size    0.084121
4        log_age    0.068642
6     laat_dummy    0.060780

Model improvement over baseline
Baseline failure rate (train): 0.3208%
Model pre